# YOLO26-FloW Evaluation

This notebook is dedicated to the evaluation of the models `yolo26m-FloW-plain.pt` and `yolo26m-FloW-sliced.pt` with and without SAHI framework and `pycocotools`.


## Plain models evaluation


In [ ]:
from ultralytics import YOLO

model_plain = YOLO("./yolo26m-FloW-plain.pt")
MODEL_PLAIN = YOLO("./yolo26m-FloW-sliced.pt")

model_plain.val(
    data="./data.yaml",
    save=True,
    save_json=True,
    project="water-plastic-detection",
    name="yolo26m-plain-eval",
)

MODEL_PLAIN.val(
    data="./data.yaml",
    save=True,
    save_json=True,
    project="water-plastic-detection",
    name="yolo26m-sliced-eval",
)

In [ ]:
import json
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def sahi_predict_and_save(
    model_path,
    coco_gt_path,
    image_dir,
    output_json_path,
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    confidence_threshold=0.001,
    device="cuda:0",
):
    """Runs SAHI sliced inference and writes COCO-format predictions to JSON."""
    coco_gt = COCO(coco_gt_path)
    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=model_path,
        confidence_threshold=confidence_threshold,
        device=device,
    )

    predictions = []
    print(f"Running SAHI inference for model {model_path}...")
    for image_id in coco_gt.getImgIds():
        img_info = coco_gt.loadImgs(image_id)[0]
        img_path = f"{image_dir}/{img_info['file_name']}"

        result = get_sliced_prediction(
            img_path,
            detection_model=detection_model,
            slice_height=slice_height,
            slice_width=slice_width,
            overlap_height_ratio=overlap_height_ratio,
            overlap_width_ratio=overlap_width_ratio,
            verbose=0,
        )

        predictions.extend(result.to_coco_predictions(image_id=image_id))

    print(f"Saving predictions to {output_json_path}...")
    with open(output_json_path, "w") as f:
        json.dump(predictions, f)

    return output_json_path


def evaluate_coco_predictions(coco_gt_path, predictions_path):
    """Evaluates COCO predictions JSON and prints COCO mAP + peak F1 metrics."""
    coco_gt = COCO(coco_gt_path)
    coco_dt = coco_gt.loadRes(predictions_path)

    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # custom peak F1 (IoU=0.5)
    import numpy as np

    precisions = coco_eval.eval["precision"][0, :, 0, 0, 2]
    recalls = np.linspace(0, 1, 101)
    valid_idx = precisions > -1
    precisions = precisions[valid_idx]
    recalls = recalls[valid_idx]

    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-16)
    peak_idx = np.argmax(f1_scores)
    peak_f1 = f1_scores[peak_idx]
    peak_precision = precisions[peak_idx]
    peak_recall = recalls[peak_idx]

    print("\n--- Peak Metrics (IoU=0.50) ---")
    print(f"Peak F1:   {peak_f1:.3f}")
    print(f"Precision: {peak_precision:.3f}")
    print(f"Recall:    {peak_recall:.3f}")

    return {
        "coco_eval": coco_eval,
        "peak_f1": float(peak_f1),
        "peak_precision": float(peak_precision),
        "peak_recall": float(peak_recall),
    }


def fix_ids_yolo(predictions_path, coco_gt_path):
    """Fixes image_id and category_id in YOLO predictions JSON for COCO evaluation."""
    with open(predictions_path, "r") as f:
        predictions = json.load(f)

    with open(coco_gt_path, "r") as f:
        coco_gt = json.load(f)

    img_name_to_id = {img["file_name"]: img["id"] for img in coco_gt["images"]}

    for pred in predictions:
        pred["category_id"] = 0
        img_name = pred["file_name"]
        pred["image_id"] = img_name_to_id.get(img_name, -1)

    with open(predictions_path, "w") as f:
        json.dump(predictions, f)


# Example usage (plain evaluation):
COCO_GT_PATH = "./FloW-img/val/annotations.json"
DIRECT_RESULTS_PLAIN_PATH = (
    "../runs/detect/water-plastic-detection/yolo26m-plain-eval/predictions.json"
)
DIRECT_RESULTS_SLICED_PATH = (
    "../runs/detect/water-plastic-detection/yolo26m-sliced-eval/predictions.json"
)

print("--- Direct YOLO result evaluation (existing JSON) ---")
fix_ids_yolo(DIRECT_RESULTS_PLAIN_PATH, COCO_GT_PATH)
fix_ids_yolo(DIRECT_RESULTS_SLICED_PATH, COCO_GT_PATH)
evaluate_coco_predictions(COCO_GT_PATH, DIRECT_RESULTS_PLAIN_PATH)
evaluate_coco_predictions(COCO_GT_PATH, DIRECT_RESULTS_SLICED_PATH)

# Example usage (SAHI):
SAHI_OUTPUT_PATH = "temp_sahi_predictions.json"

sahi_predict_and_save(
    model_path="./yolo26m-FloW-plain.pt",
    coco_gt_path=COCO_GT_PATH,
    image_dir="./FloW-img/val/images",
    output_json_path=SAHI_OUTPUT_PATH,
)
evaluate_coco_predictions(COCO_GT_PATH, SAHI_OUTPUT_PATH)

sahi_predict_and_save(
    model_path="./yolo26m-FloW-sliced.pt",
    coco_gt_path=COCO_GT_PATH,
    image_dir="./FloW-img/val/images",
    output_json_path=SAHI_OUTPUT_PATH,
)
evaluate_coco_predictions(COCO_GT_PATH, SAHI_OUTPUT_PATH)
